<a href="https://colab.research.google.com/github/SebGalvaleira/mineria-datos-ii-parcial/blob/main/notebooks/e2/E2_06_serving_astradb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E2_06 · Serving en AstraDB (Cassandra)

**Dónde estamos:** Landing → Bronze → Silver → Gold → **Serving**. Gold ya tiene el mart `org_daily_usage_by_service` validado (11.050 filas · 147.433,98 USD). Ahora lo publicamos en AstraDB para que FinOps lo consulte sin Spark.

**Diseño query-first:** primero se definen las consultas y después las tablas. Cassandra no hace joins y solo filtra eficientemente por la clave.

| Consulta (FinOps) | Tabla | Partition key | Clustering |
|---|---|---|---|
| Q1 · ¿Cuánto gastó la org X por servicio y por día en un rango de fechas? | `org_daily_usage_by_service` | `(org_id, usage_month)` | `usage_date DESC, service` |
| Q2 · ¿Qué orgs/servicios tuvieron costos anómalos el día D? | `cost_anomalies_by_date` | `usage_date` | `org_id, service` |

In [ ]:
!pip install -q "pyspark>=4.0,<4.1" cassandra-driver
import os, shutil, subprocess
os.environ.pop('JAVA_HOME', None)

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
LAKE_DRIVE = Path('/content/drive/MyDrive/cpa_datalake')   # respaldo persistente
LAKE       = Path('/content/cpa_datalake')                  # copia de trabajo (disco local de Colab)

RUTAS = {z: LAKE / z for z in ['landing', 'stream_input', 'bronze', 'silver',
                               'gold', 'quarantine', 'checkpoints', 'logs']}

def restaurar(*rutas):
    """Copia desde Drive SOLO las carpetas que este notebook necesita (si no están ya en local)."""
    for r in rutas:
        origen, destino = LAKE_DRIVE / r, LAKE / r
        if origen.exists() and not destino.exists():
            shutil.copytree(origen, destino)
            print('✓ restaurado:', r)

def guardar_en_drive(*rutas):
    """Sincroniza en Drive SOLO las carpetas indicadas."""
    for r in rutas:
        origen, destino = LAKE / r, LAKE_DRIVE / r
        destino.mkdir(parents=True, exist_ok=True)
        subprocess.run(['rsync', '-a', '--delete', f'{origen}/', f'{destino}/'], check=True)
        print('✓ sincronizado:', r)

from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder.appName('cloud-provider-analytics')
         .config('spark.sql.session.timeZone', 'UTC')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.sql.sources.partitionOverwriteMode', 'dynamic')
         .getOrCreate())
print('Spark', spark.version)

Mounted at /content/drive
Spark 4.0.5


In [ ]:
#restaurar
# solo necesitamos gold (lo que publicamos) y logs (donde dejamos la evidencia)
import json, time
import pandas as pd
restaurar('gold', 'logs')
for p in RUTAS.values():
    p.mkdir(parents=True, exist_ok=True)

✓ restaurado: gold
✓ restaurado: logs


In [ ]:
#conexion
# el token sale de los Secrets de Colab y el bundle de Drive: nada sensible queda escrito en el notebook
# LOCAL_QUORUM: Astra exige quórum para escribir; también lo usamos para leer, así siempre vemos lo último escrito
from google.colab import userdata
from cassandra.cluster import Cluster, ExecutionProfile, EXEC_PROFILE_DEFAULT
from cassandra.auth import PlainTextAuthProvider
from cassandra import ConsistencyLevel

SCB = next(Path('/content/drive/MyDrive/cpa_secrets').glob('secure-connect-*.zip'))
KEYSPACE = 'finops'

perfil = ExecutionProfile(consistency_level=ConsistencyLevel.LOCAL_QUORUM, request_timeout=60)
cluster = Cluster(cloud={'secure_connect_bundle': str(SCB)},
                  auth_provider=PlainTextAuthProvider('token', userdata.get('ASTRA_DB_TOKEN')),
                  execution_profiles={EXEC_PROFILE_DEFAULT: perfil},
                  protocol_version=4)
session = cluster.connect(KEYSPACE)
print('Conectado a', SCB.name, '· Cassandra', session.execute('SELECT release_version FROM system.local').one()[0])

Conectado a secure-connect-cloud-provider-analytics.zip · Cassandra 4.0.11.0-6819aa9c7f6c


In [ ]:
#crear las tablas
# en cassandra la clave primaria es el diseño
# Q1: partición (org_id, usage_month) -> como máximo 31 días x 6 servicios = 186 filas, acotada aunque pasen años
#     clustering usage_date DESC: los días quedan ordenados en disco, el rango de fechas es una lectura secuencial
# Q2: partición usage_date -> todas las anomalías del día juntas, una sola lectura
CQL_ESQUEMA = """
CREATE TABLE IF NOT EXISTS finops.org_daily_usage_by_service (
    org_id text, usage_month text, usage_date date, service text,
    daily_cost_usd double, requests double, cpu_hours double, storage_gb_hours double,
    genai_tokens bigint, carbon_kg double,
    eventos int, eventos_costo_anomalo int, eventos_pico int,
    industry text, plan_tier text, updated_at timestamp,
    PRIMARY KEY ((org_id, usage_month), usage_date, service)
) WITH CLUSTERING ORDER BY (usage_date DESC, service ASC);

CREATE TABLE IF NOT EXISTS finops.cost_anomalies_by_date (
    usage_date date, org_id text, service text,
    daily_cost_usd double, eventos int, eventos_costo_anomalo int, eventos_pico int,
    industry text, plan_tier text, updated_at timestamp,
    PRIMARY KEY ((usage_date), org_id, service)
);
"""
for sentencia in [s.strip() for s in CQL_ESQUEMA.split(';') if s.strip()]:
    session.execute(sentencia)
(RUTAS['logs'] / 'astra_schema.cql').write_text(CQL_ESQUEMA.strip() + '\n')   # va al repo como script CQL

print([r.table_name for r in session.execute(
    "SELECT table_name FROM system_schema.tables WHERE keyspace_name = 'finops'")])

['cost_anomalies_by_date', 'org_daily_usage_by_service']


In [ ]:
#preparar gold para cassandra
# industry y plan_tier ya vienen dentro del mart (desnormalizados en gold) porque cassandra no hace joins
# None -> UNSET_VALUE: en cassandra escribir un null genera un "tombstone" (marca de borrado). con UNSET la celda no se escribe
from cassandra.query import UNSET_VALUE

gold = spark.read.parquet(str(RUTAS['gold'] / 'org_daily_usage_by_service'))

serving = (gold
    .withColumn('usage_date', F.col('event_date').cast('date'))
    .withColumn('usage_month', F.date_format('usage_date', 'yyyy-MM'))
    .select('org_id', 'usage_month', 'usage_date', 'service',
            *[F.col(c).cast('double').alias(c) for c in ['daily_cost_usd', 'requests', 'cpu_hours', 'storage_gb_hours']],
            F.col('genai_tokens').cast('long').alias('genai_tokens'),
            F.col('carbon_kg').cast('double').alias('carbon_kg'),
            *[F.col(c).cast('int').alias(c) for c in ['eventos', 'eventos_costo_anomalo', 'eventos_pico']],
            'industry', 'plan_tier', 'updated_at'))
anomalias = serving.filter((F.col('eventos_pico') > 0) | (F.col('eventos_costo_anomalo') > 0))

COLS_USO  = serving.columns
COLS_ANOM = ['usage_date', 'org_id', 'service', 'daily_cost_usd', 'eventos',
             'eventos_costo_anomalo', 'eventos_pico', 'industry', 'plan_tier', 'updated_at']

def a_filas(df, cols):
    """Spark -> lista de tuplas para el driver."""
    return [tuple(UNSET_VALUE if v is None else v for v in r) for r in df.select(*cols).collect()]

filas_uso, filas_anom = a_filas(serving, COLS_USO), a_filas(anomalias, COLS_ANOM)
max_part = serving.groupBy('org_id', 'usage_month').count().agg(F.max('count')).first()[0]
print('A cargar · uso:', len(filas_uso), 'filas · anomalías:', len(filas_anom), 'filas')
print('Partición más grande de Q1 (org, mes):', max_part, 'filas (tope teórico 186)')

A cargar · uso: 11050 filas · anomalías: 256 filas
Partición más grande de Q1 (org, mes): 147 filas (tope teórico 186)


In [ ]:
#cargar t1 2 veces (idempotencia)
# INSERT en cassandra es un upsert: si la clave ya existe, sobrescribe. por eso recargar no duplica
# execute_concurrent: manda muchas escrituras en paralelo desde el driver (el conector spark-cassandra todavía no soporta spark 4)
from cassandra.concurrent import execute_concurrent_with_args

def upsert(tabla, cols, filas):
    stmt = session.prepare(f"INSERT INTO {tabla} ({', '.join(cols)}) VALUES ({', '.join('?' * len(cols))})")
    t0 = time.time()
    execute_concurrent_with_args(session, stmt, filas, concurrency=64, raise_on_first_error=True)
    return round(time.time() - t0, 1)

def contar(tabla):
    return session.execute(f'SELECT COUNT(*) FROM {tabla}', timeout=120).one()[0]

seg1 = upsert('org_daily_usage_by_service', COLS_USO, filas_uso);  n1 = contar('org_daily_usage_by_service')
seg2 = upsert('org_daily_usage_by_service', COLS_USO, filas_uso);  n2 = contar('org_daily_usage_by_service')
print(f'Carga 1: {n1} filas en {seg1}s · Carga 2 (repetida): {n2} filas en {seg2}s')
assert n1 == n2 == len(filas_uso) == 11_050, 'La tabla no coincide con Gold'
print('Recargar no duplica: el upsert por clave natural es idempotente')

Carga 1: 11050 filas en 11.5s · Carga 2 (repetida): 11050 filas en 11.8s
Recargar no duplica: el upsert por clave natural es idempotente


In [ ]:
#cargar q2 (reemplazo por fecha)
# si al recalcular gold una anomalía desaparece, un upsert no la borraría
# por eso hacemos lo mismo que en gold: reemplazar la partición entera del día (borrar + insertar)
fechas = [(r[0],) for r in serving.select('usage_date').distinct().collect()]
borrar = session.prepare('DELETE FROM cost_anomalies_by_date WHERE usage_date = ?')
execute_concurrent_with_args(session, borrar, fechas, concurrency=32, raise_on_first_error=True)
seg = upsert('cost_anomalies_by_date', COLS_ANOM, filas_anom)
n = contar('cost_anomalies_by_date')
print(f'Anomalías: {n} filas en {seg}s · {len(fechas)} días reemplazados')
assert n == len(filas_anom)

Anomalías: 256 filas en 0.3s · 60 días reemplazados


In [ ]:
#consulta 1
# siempre se consulta con la partition key completa (org + mes) y un rango sobre la clustering (fecha)
CQL_Q1 = """SELECT usage_date, service, daily_cost_usd, eventos, eventos_pico
FROM finops.org_daily_usage_by_service
WHERE org_id = %s AND usage_month = %s AND usage_date >= %s AND usage_date <= %s"""

org_top = gold.groupBy('org_id').agg(F.sum('daily_cost_usd').alias('c')).orderBy(F.desc('c')).first()['org_id']
PARAMS_Q1 = (org_top, '2025-08', '2025-08-01', '2025-08-07')

q1 = pd.DataFrame(list(session.execute(CQL_Q1, PARAMS_Q1)))
q1['usage_date'] = q1.usage_date.astype(str)
print('Org con mayor gasto:', org_top, '· 1 al 7 de agosto')
display(q1.pivot_table(index='usage_date', columns='service', values='daily_cost_usd', aggfunc='sum').round(2))

esperado = (gold.filter((F.col('org_id') == org_top) & F.col('event_date').between('2025-08-01', '2025-08-07'))
                .agg(F.sum('daily_cost_usd')).first()[0])
assert abs(q1.daily_cost_usd.sum() - esperado) < 1e-3, 'AstraDB no coincide con Gold'
print(f'Total {q1.daily_cost_usd.sum():.4f} USD = Gold ✓')

Org con mayor gasto: org_53lc58dr · 1 al 7 de agosto


service,analytics,compute,database,genai,networking,storage
usage_date,,,,,,
2025-08-01,NaN,NaN,1.58,15.64,4.31,NaN
2025-08-02,32.76,NaN,1.56,NaN,2.26,4.84
2025-08-03,25.46,12.63,NaN,19.29,1.40,2.39
2025-08-04,30.01,8.27,25.49,45.23,NaN,NaN
2025-08-05,8.46,37.20,31.77,NaN,0.07,5.49
2025-08-06,13.74,29.08,NaN,15.97,0.23,NaN
2025-08-07,22.05,9.82,6.39,NaN,NaN,5.79


Total 419.1986 USD = Gold ✓


In [ ]:
#consulta 2
CQL_Q2 = """SELECT org_id, service, daily_cost_usd, eventos_pico, eventos_costo_anomalo, industry, plan_tier
FROM finops.cost_anomalies_by_date WHERE usage_date = %s"""

dia_top = str(anomalias.groupBy('usage_date').count().orderBy(F.desc('count')).first()['usage_date'])
q2 = pd.DataFrame(list(session.execute(CQL_Q2, (dia_top,)))).sort_values('daily_cost_usd', ascending=False)
print('Día con más anomalías:', dia_top)
display(q2)
assert len(q2) == anomalias.filter(F.col('usage_date') == dia_top).count()

Día con más anomalías: 2025-08-23


,org_id,service,daily_cost_usd,eventos_pico,eventos_costo_anomalo,industry,plan_tier
0,org_1t2tala7,compute,257.3640,1,0,media,enterprise
9,org_teiyzcot,compute,224.1130,1,0,healthcare,pro
10,org_x7eedtjv,compute,211.2481,1,0,energy,pro
8,org_pja1wj0t,compute,205.8369,1,0,education,standard
1,org_9t84azyt,database,147.5435,1,0,fintech,enterprise
4,org_dhylurtp,database,18.2029,0,1,fintech,pro
2,org_c11ertj5,compute,14.4087,0,1,retail,pro
6,org_kdgigatj,database,6.3949,0,1,gaming,standard
3,org_dbdw2pcn,database,5.0808,0,1,retail,standard
5,org_hv3a3zmf,networking,0.3103,0,1,media,pro


In [ ]:
#consulta rechazada (porque es query - first)
# filtrar por una columna que no es clave obligaría a recorrer todas las particiones: cassandra lo rechaza
try:
    session.execute("SELECT * FROM finops.org_daily_usage_by_service WHERE service = 'genai'")
except Exception as e:
    print('Rechazada como se esperaba →', type(e).__name__, ':', str(e)[:160])

Rechazada como se esperaba → InvalidRequest : Error from server: code=2200 [Invalid query] message="PRIMARY KEY column "service" cannot be restricted as preceding column "usage_date" is not restricted"


In [ ]:
#evidencia y cierre
q1.to_csv(RUTAS['logs'] / 'astra_q1.csv', index=False)
q2.to_csv(RUTAS['logs'] / 'astra_q2.csv', index=False)

# las mismas consultas con valores concretos, para pegarlas en el CQL Console de Astra y sacar la captura
(RUTAS['logs'] / 'astra_consultas.cql').write_text(
    f"-- Q1: gasto diario por servicio de una org en un rango\n"
    f"SELECT usage_date, service, daily_cost_usd, eventos, eventos_pico FROM finops.org_daily_usage_by_service\n"
    f"WHERE org_id = '{org_top}' AND usage_month = '2025-08' AND usage_date >= '2025-08-01' AND usage_date <= '2025-08-07';\n\n"
    f"-- Q2: anomalías de costo de un día\n"
    f"SELECT org_id, service, daily_cost_usd, eventos_pico, eventos_costo_anomalo FROM finops.cost_anomalies_by_date\n"
    f"WHERE usage_date = '{dia_top}';\n")

json.dump({'tabla_uso': n2, 'tabla_anomalias': n, 'segundos_carga_1': seg1, 'segundos_carga_2': seg2,
           'particion_max_q1': max_part, 'org_q1': org_top, 'dia_q2': dia_top},
          open(RUTAS['logs'] / 'astra_carga.json', 'w'), indent=2)
cluster.shutdown()
guardar_en_drive('logs')
print(open(RUTAS['logs'] / 'astra_consultas.cql').read())

✓ sincronizado: logs
-- Q1: gasto diario por servicio de una org en un rango
SELECT usage_date, service, daily_cost_usd, eventos, eventos_pico FROM finops.org_daily_usage_by_service
WHERE org_id = 'org_53lc58dr' AND usage_month = '2025-08' AND usage_date >= '2025-08-01' AND usage_date <= '2025-08-07';

-- Q2: anomalías de costo de un día
SELECT org_id, service, daily_cost_usd, eventos_pico, eventos_costo_anomalo FROM finops.cost_anomalies_by_date
WHERE usage_date = '2025-08-23';



## Cierre: serving en AstraDB

**Dónde estamos:** el pipeline llega a la capa de **serving**. El mart de Gold quedó publicado en AstraDB (keyspace `finops`), y FinOps lo puede consultar en milisegundos sin levantar Spark.

**Diseño query-first:** cada tabla responde exactamente a una pregunta de FinOps (§1.2 de la Entrega 1).

| Consulta | Tabla | Partition key | Clustering | Por qué |
|---|---|---|---|---|
| Q1 · Gasto diario por servicio de una org en un rango | `org_daily_usage_by_service` | `(org_id, usage_month)` | `usage_date DESC, service` | El bucket mensual acota la partición (máx. real 147 filas, tope 31 × 6 = 186). La fecha como clustering hace que el rango sea una lectura secuencial |
| Q2 · Anomalías de costo de un día | `cost_anomalies_by_date` | `usage_date` | `org_id, service` | Todas las anomalías del día en una sola partición (≈ 4 filas por día en promedio) |

**Resultados**
- `org_daily_usage_by_service`: **11.050 filas**, igual que Gold. La carga se repitió dos veces y el conteo no cambió.
- `cost_anomalies_by_date`: **256 filas** (260 eventos anómalos agrupados por org × día × servicio).
- Q1 (`org_53lc58dr`, 1–7/08): **419,1986 USD**, idéntico a Gold.
- Q2 (23/08, el día con más anomalías): 11 combinaciones, 5 picos de compute/database y 6 costos negativos.
- Una consulta por `service` sin la partition key es **rechazada** (`InvalidRequest`). Cassandra solo responde las consultas para las que se diseñó la tabla.

**Decisiones**
- **Idempotencia:** `INSERT` es upsert por clave natural. En Q2, como una anomalía puede desaparecer al recalcular, se reemplaza la partición del día completa (borrar + insertar), igual que el overwrite por fecha de Gold.
- **Desnormalización:** `industry` y `plan_tier` viajan dentro de cada fila porque Cassandra no hace joins.
- **Nulos → `UNSET_VALUE`:** escribir `null` genera tombstones. Las métricas que no aplican (por ejemplo, `carbon_kg` en v1) simplemente no se escriben.
- **Consistencia `LOCAL_QUORUM`:** la exige Astra para escribir, y al leer garantiza ver lo último escrito.
- **Carga con el driver de Python** (`execute_concurrent`, 64 escrituras en paralelo, ≈ 12 s): el conector spark-cassandra todavía no soporta Spark 4. En producción se haría desde `foreachBatch`, con la misma función de upsert.
- **Seguridad:** el token vive en los Secrets de Colab y el bundle en Drive (`cpa_secrets/`). Ninguno de los dos está en el notebook ni en el repo.

**Evidencia:** `logs/astra_schema.cql`, `logs/astra_consultas.cql`, `logs/astra_carga.json`, `logs/astra_q1.csv`, `logs/astra_q2.csv` y las capturas del CQL Console.